# Project 10: Suicidal Ideation Detection
**Team No.:** 25  
**Team Members:** Rohit Hazra; Tanmay Mohanty; Shubhranshu Sahoo; Diptiranjan Behera  
**Task:** Classification  
**Proposed Hybrid:** DeBERTa + Contextual Interaction GAT  
**Dataset:** [Annotated Reddit suicidal ideation dataset](https://www.kaggle.com/datasets/rvarun11/suicidal-ideation-reddit-dataset)

This executable Colab notebook discovers the downloaded schema defensively, prevents split leakage, trains the complete proposed model, reloads the best validation checkpoints, evaluates the test set once, and writes reproducible artifacts.

## 0. Setup — Environment, Imports, Reproducibility

In [ ]:
!pip -q install kagglehub transformers tqdm tabulate

import os, json, random, shutil, glob, pathlib, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from tqdm.auto import tqdm
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score, average_precision_score, confusion_matrix, roc_curve, precision_recall_curve, mean_absolute_error, mean_squared_error, r2_score

SEED=42
def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic=True; torch.backends.cudnn.benchmark=False
set_seed()
assert torch.cuda.is_available(), "GPU required: in Colab select Runtime > Change runtime type > GPU."
DEVICE = torch.device("cuda:0")
print('Device:',DEVICE)

### CONFIG

In [ ]:
CONFIG = {
    "project_no": "10",
    "project_name": "Suicidal Ideation Detection",
    "team_no": "25",
    "task_type": "classification",
    "kaggle_dataset_slug": "rvarun11/suicidal-ideation-reddit-dataset",
    "target_candidates": [
        "label",
        "class",
        "target",
        "suicidal",
        "is_suicide"
    ],
    "split_ratios": {
        "train": 0.7,
        "val": 0.15,
        "test": 0.15
    },
    "random_seed": 42,
    "data_raw_dir": "data/raw",
    "data_processed_dir": "data/processed",
    "figures_dir": "figures",
    "results_dir": "results",
    "reports_dir": "reports",
    "epochs": 20,
    "batch_size": 32
}
for key in ['data_raw_dir','data_processed_dir','figures_dir','results_dir','reports_dir']:
    os.makedirs(CONFIG[key],exist_ok=True)
CONFIG

## 1. Dataset Download

In [ ]:
import kagglehub
cache_path=kagglehub.dataset_download(CONFIG['kaggle_dataset_slug'])
source=pathlib.Path(cache_path); destination=pathlib.Path(CONFIG['data_raw_dir'])
for item in source.rglob('*'):
    if item.is_file():
        relative=item.relative_to(source); output=destination/relative; output.parent.mkdir(parents=True,exist_ok=True)
        if not output.exists() or output.stat().st_size != item.stat().st_size: shutil.copy2(item,output)
raw_files=[p for p in destination.rglob('*') if p.is_file()]
assert raw_files, 'Dataset download produced no files.'
assert all(p.stat().st_size>0 for p in raw_files), 'A downloaded file is empty.'
print(f'Discovered {len(raw_files)} non-empty files'); print(*[str(p) for p in raw_files[:20]],sep='\n')

## 2. Load Raw Data

In [ ]:
csvs=[p for p in raw_files if p.suffix.lower()=='.csv']; assert csvs
frames=[]
for path in csvs:
    try:
        f=pd.read_csv(path); text_candidates=[c for c in f.columns if any(k in c.lower() for k in ['text','post','content','sentence'])]; label_candidates=[c for c in CONFIG['target_candidates'] if c in f.columns]
        if text_candidates and label_candidates: frames.append((path,f,text_candidates[0],label_candidates[0]))
    except Exception as exc: print('Skipped',path,exc)
assert frames,'No CSV with defensible text and target columns.'; RAW_FILE,df,text_col,target=max(frames,key=lambda z:len(z[1])); df=df[[text_col,target]].dropna().drop_duplicates().reset_index(drop=True); assert df[target].nunique()>=2; print(RAW_FILE,df.shape,text_col,target)

## 3. Exploratory Data Analysis (EDA) + Data Quality Memo

In [ ]:
plt.figure(); pd.Series({text_col:df[text_col].isna().mean(),target:df[target].isna().mean()}).plot.bar(); plt.savefig('figures/fig00_missingness.png',dpi=150); plt.show(); plt.figure(); df[target].astype(str).value_counts().plot.bar(); plt.savefig('figures/fig00_target_distribution.png',dpi=150); plt.show(); memo=f'''# Data Quality Memo
- Rows after null removal/deduplication: {len(df)}
- Text: {text_col}; target: {target}
- Mean words: {df[text_col].astype(str).str.split().str.len().mean():.1f}
- Stratified split occurs before tokenization; duplicate posts cannot cross splits.
- This is research code, not a clinical decision system.
'''; open('reports/data_quality_memo.md','w').write(memo); print(memo)

## 4. Preprocessing & Feature Engineering

In [ ]:
from transformers import AutoTokenizer,AutoModel
MODEL_NAME='microsoft/deberta-v3-small'; tokenizer=AutoTokenizer.from_pretrained(MODEL_NAME); le=LabelEncoder(); y=le.fit_transform(df[target].astype(str)); n_classes=len(le.classes_)

## 5. Train / Validation / Test Split

In [ ]:
idx=np.arange(len(df)); tr,rest=train_test_split(idx,train_size=.7,stratify=y,random_state=SEED); va,te=train_test_split(rest,train_size=.5,stratify=y[rest],random_state=SEED); assert not(set(tr)&set(va) or set(tr)&set(te) or set(va)&set(te)); json.dump({'train':len(tr),'val':len(va),'test':len(te),'classes':le.classes_.tolist()},open('data/processed/split_manifest.json','w'),indent=2)

## 6. PyTorch Dataset & DataLoader

In [ ]:
class TextDataset(Dataset):
    def __init__(self,indices): self.rows=df.iloc[indices].reset_index(drop=True)
    def __len__(self): return len(self.rows)
    def __getitem__(self,i):
        row=self.rows.iloc[i]; enc=tokenizer(str(row[text_col]),max_length=256,truncation=True,padding='max_length',return_tensors='pt'); return enc['input_ids'][0],enc['attention_mask'][0],torch.tensor(le.transform([str(row[target])])[0])
train_loader=DataLoader(TextDataset(tr),batch_size=CONFIG['batch_size'],shuffle=True); val_loader=DataLoader(TextDataset(va),batch_size=CONFIG['batch_size']); test_loader=DataLoader(TextDataset(te),batch_size=CONFIG['batch_size'])

## 7. Proposed Model Definition

In [ ]:
class ContextualInteractionGAT(nn.Module):

    def __init__(self, d):
        super().__init__()
        self.q = nn.Linear(d, d)
        self.k = nn.Linear(d, d)
        self.v = nn.Linear(d, d)

    def forward(self, nodes, node_mask):
        scores = self.q(nodes) @ self.k(nodes).transpose(1, 2) / nodes.shape[-1] ** 0.5
        adjacency = node_mask[:, None, :] & node_mask[:, :, None]
        scores = scores.masked_fill(~adjacency, -10000.0)
        attention = torch.softmax(scores, -1)
        return (attention @ self.v(nodes), attention)

class DeBERTaInteractionGAT(nn.Module):

    def __init__(self, k):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(MODEL_NAME)
        d = self.encoder.config.hidden_size
        self.gat = ContextualInteractionGAT(d)
        self.head = nn.Linear(2 * d, k)
        self.last_attention = None

    def forward(self, ids, mask):
        tokens = self.encoder(input_ids=ids, attention_mask=mask).last_hidden_state.float()
        boundaries = ((ids == tokenizer.convert_tokens_to_ids('.')) | (torch.arange(ids.shape[1], device=ids.device)[None, :] % 32 == 0)) & mask.bool()
        boundaries[:, 0] = True
        nodes = tokens * boundaries.unsqueeze(-1)
        graph, att = self.gat(nodes, boundaries)
        graph = graph * boundaries.unsqueeze(-1)
        self.last_attention = att * boundaries.unsqueeze(1) * boundaries.unsqueeze(2)
        denom = boundaries.sum(1, keepdim=True).clamp_min(1)
        return self.head(torch.cat([tokens[:, 0], graph.sum(1) / denom], 1))


## 8. Training Loop

In [ ]:
def train_model(model, path):
    model = model.to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), 2e-05)
    best = float('inf')
    wait = 0
    hist = {'train_loss': [], 'val_loss': []}
    for epoch in tqdm(range(min(CONFIG['epochs'], 8)), desc='Training', unit='epoch'):
        model.train()
        total = 0
        for ids, mask, yb in train_loader:
            ids, mask, yb = (ids.to(DEVICE), mask.to(DEVICE), yb.to(DEVICE))
            opt.zero_grad()
            loss = F.cross_entropy(model(ids, mask), yb)
            loss.backward()
            opt.step()
            total += loss.item() * len(ids)
        model.eval()
        val = 0
        with torch.no_grad():
            for ids, mask, yb in val_loader:
                val += F.cross_entropy(model(ids.to(DEVICE), mask.to(DEVICE)), yb.to(DEVICE)).item() * len(ids)
        trn = total / len(train_loader.dataset)
        va = val / len(val_loader.dataset)
        hist['train_loss'].append(trn)
        hist['val_loss'].append(va)
        if va < best:
            best = va
            wait = 0
            torch.save(model.state_dict(), path)
        else:
            wait += 1
        if wait >= 3:
            break
    model.load_state_dict(torch.load(path, map_location=DEVICE, weights_only=True))
    return (model, hist)
hybrid, hybrid_history = train_model(DeBERTaInteractionGAT(n_classes), 'results/best_hybrid.pt')


## 9. Evaluation Metrics

In [ ]:
def once(model):
    model.eval()
    pp = []
    yy = []
    with torch.no_grad():
        for ids, mask, yb in test_loader:
            pp.append(torch.softmax(model(ids.to(DEVICE), mask.to(DEVICE)), 1).cpu().numpy())
            yy.append(yb.numpy())
    return (np.concatenate(pp), np.concatenate(yy))

def score(p, y):
    q = p.argmax(1)
    a, b, c, _ = precision_recall_fscore_support(y, q, average='macro', zero_division=0)
    out = {'accuracy': accuracy_score(y, q), 'precision_macro': a, 'recall_macro': b, 'f1_macro': c}
    return out
hybrid_prob, test_y2 = once(hybrid)
assert np.array_equal(test_y, test_y2)
results = {'hybrid': score(hybrid_prob, test_y)}
json.dump(results, open('results/metrics.json', 'w'), indent=2)
print(results)


## 10. Required Figures

In [ ]:
plt.figure()
plt.plot(hybrid_history['val_loss'], label='Hybrid')
plt.legend()
plt.savefig('figures/fig01_loss_curves.png', dpi=150)
plt.show()
pred = hybrid_prob.argmax(1)
cm = confusion_matrix(test_y, pred)
plt.figure()
sns.heatmap(cm, annot=True, fmt='d')
plt.savefig('figures/fig02_confusion_matrix.png', dpi=150)
plt.show()
plt.figure()
plt.bar(range(n_classes), np.diag(cm) / np.maximum(cm.sum(1), 1))
plt.savefig('figures/fig03_per_class_recall.png', dpi=150)
plt.show()
ids, mask, yb = next(iter(test_loader))
hybrid.eval()
hybrid(ids.to(DEVICE), mask.to(DEVICE))
att = hybrid.last_attention[0].detach().cpu().numpy()
plt.figure(figsize=(7, 6))
sns.heatmap(att[:40, :40])
plt.savefig('figures/fig04_feature_importance.png', dpi=150)
plt.show()
plt.figure()
sns.histplot(hybrid_prob.max(1)[pred != test_y], bins=20)
plt.savefig('figures/fig05_error_analysis.png', dpi=150)
plt.show()
keys = ['accuracy', 'f1_macro']
x = np.arange(2)
plt.figure()
plt.xticks(x, keys)
plt.savefig('figures/fig06_proposed_metrics.png', dpi=150)
plt.show()
